# Elastic Search

In [14]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Vérifier qu'on est bien à la racine
RACINE = Path(os.getcwd()).parent

print(RACINE)
assert (RACINE / ".env").exists(), \
    f"Lance Jupyter depuis la racine du projet. Dossier actuel : {RACINE}"

load_dotenv(RACINE / ".env")

print(f"Racine    : {RACINE}")
print(f"ELASTIC   : {os.getenv('ELASTIC_HOST')}")
print(f"MONGO     : {os.getenv('MONGO_HOST')}")
print(f"POSTGRES  : {os.getenv('POSTGRES_HOST')}")

/home/thierry/code/thcarole1/projects/job-market-pipeline-clean
Racine    : /home/thierry/code/thcarole1/projects/job-market-pipeline-clean
ELASTIC   : localhost
MONGO     : 127.0.0.1
POSTGRES  : localhost


In [15]:
# storage/elasticsearch/index.py
"""
Gestion de l'index Elasticsearch pour le projet Job Market.

Rôle : indexer les offres normalisées pour la recherche full-text.
       Elasticsearch reçoit une copie des données normalisées,
       optimisée pour la recherche — pas pour le stockage principal.

Flux :
    data/processed/normalise/*.json
            ↓
    index.py (mapping + indexation)
            ↓
    Elasticsearch index "offres"
"""

import json
import os
import logging
from pathlib import Path
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk, BulkIndexError

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s"
)

# RACINE = Path(__file__).parent.parent.parent
NOM_INDEX = "offres"


# ─────────────────────────────────────────────────────────────
# CONNEXION
# ─────────────────────────────────────────────────────────────

def connecter_elasticsearch() -> Elasticsearch:
    """
    Crée et retourne une connexion Elasticsearch.
    Vérifie que le cluster est accessible avant de retourner le client.
    """
    host = os.getenv("ELASTIC_HOST", "localhost")
    port = int(os.getenv("ELASTIC_PORT", 9200))

    client = Elasticsearch(
        f"http://{host}:{port}",
        request_timeout=30,   # secondes avant abandon
        retry_on_timeout=True,
        max_retries=3,        # nombre de tentatives automatiques
    )

    # Vérifier que le cluster répond
    if not client.ping():
        raise ConnectionError(
            f"Impossible de joindre Elasticsearch sur {host}:{port}. "
            f"Vérifiez que le conteneur Docker est démarré."
        )

    info = client.info()
    logging.info(
        f"Connecté à Elasticsearch "
        f"v{info['version']['number']} "
        f"sur {host}:{port}"
    )

    return client


# ─────────────────────────────────────────────────────────────
# MAPPING DE L'INDEX
# ─────────────────────────────────────────────────────────────

# Le mapping définit le type de chaque champ.
# text     → analysé linguistiquement — pour la recherche full-text
#            "engineers" et "engineer" sont considérés identiques
# keyword  → valeur exacte — pour les filtres et agrégations
#            "CDI" reste "CDI", pas analysé
# integer  → nombre entier — pour les filtres de plage (salaire >= 40000)
# date     → date — pour le tri et les filtres temporels

MAPPING = {
    "mappings": {
        "properties": {

            # ── Identifiants ──────────────────────────────────
            "id":     {"type": "keyword"},
            "source": {"type": "keyword"},
            "url":    {"type": "keyword"},

            # ── Champs textuels analysés ──────────────────────
            # Ces champs alimentent la recherche full-text.
            # L'analyseur "french" gère les accents et la conjugaison.
            "titre": {
                "type":     "text",
                "analyzer": "french",
                "fields": {
                    "keyword": {"type": "keyword"}  # pour le tri exact
                }
            },
            "description": {
                "type":     "text",
                "analyzer": "french"
            },
            "competences": {
                "type":     "text",
                "analyzer": "french",
                "fields": {
                    "keyword": {"type": "keyword"}
                }
            },
            "missions": {
                "type":     "text",
                "analyzer": "french"
            },

            # ── Champs exacts (filtres) ────────────────────────
            # Ces champs ne sont pas analysés — valeur exacte uniquement.
            "entreprise":          {"type": "keyword"},
            "localisation_ville":  {"type": "keyword"},
            "localisation_dept":   {"type": "keyword"},
            "type_contrat":        {"type": "keyword"},
            "teletravail":         {"type": "keyword"},
            "secteur":             {"type": "keyword"},
            "rome_code":           {"type": "keyword"},

            # ── Champs numériques ──────────────────────────────
            # Pour les filtres de plage : salaire >= 40000
            "salaire_min":    {"type": "integer"},
            "salaire_max":    {"type": "integer"},
            "experience_min": {"type": "integer"},

            # ── Coordonnées GPS ───────────────────────────────
            # geo_point permet des recherches géographiques
            # "offres dans un rayon de 50km autour de Paris"
            "localisation": {"type": "geo_point"},

            # ── Dates ─────────────────────────────────────────
            "date_publication": {"type": "date"},
            "date_extraction":  {"type": "date"},
        }
    },
    "settings": {
        "number_of_shards":   1,  # 1 shard suffit pour un projet de formation
        "number_of_replicas": 0,  # 0 replica en développement local
    }
}


# ─────────────────────────────────────────────────────────────
# CRÉATION DE L'INDEX
# ─────────────────────────────────────────────────────────────

def creer_index(client: Elasticsearch, nom_index: str = NOM_INDEX):
    """
    Crée l'index Elasticsearch avec le mapping défini.
    Si l'index existe déjà, ne fait rien — idempotent.
    """
    if client.indices.exists(index=nom_index):
        logging.info(f"Index '{nom_index}' existe déjà — pas de recréation.")
        return

    client.indices.create(index=nom_index, body=MAPPING)
    logging.info(f"Index '{nom_index}' créé avec succès.")


def supprimer_index(client: Elasticsearch, nom_index: str = NOM_INDEX):
    """
    Supprime l'index — utile pour repartir proprement si le mapping change.
    """
    if client.indices.exists(index=nom_index):
        client.indices.delete(index=nom_index)
        logging.info(f"Index '{nom_index}' supprimé.")
    else:
        logging.info(f"Index '{nom_index}' n'existe pas — rien à supprimer.")


# ─────────────────────────────────────────────────────────────
# PRÉPARATION DES DOCUMENTS
# ─────────────────────────────────────────────────────────────

def preparer_document(offre: dict) -> dict:
    """
    Prépare une offre normalisée pour l'indexation Elasticsearch.

    Transformations appliquées :
    - Fusion latitude/longitude en geo_point si disponibles
    - Nettoyage des champs None (Elasticsearch préfère les champs absents)
    - Utilisation de l'id métier comme _id Elasticsearch
    """
    doc = {}

    # Copier tous les champs non None
    for cle, valeur in offre.items():
        if valeur is not None and valeur != "" and valeur != []:
            doc[cle] = valeur

    # Fusionner latitude et longitude en geo_point si disponibles
    lat = offre.get("latitude")
    lon = offre.get("longitude")
    if lat and lon:
        doc["localisation"] = {"lat": lat, "lon": lon}
        # Supprimer les champs séparés — geo_point les remplace
        doc.pop("latitude", None)
        doc.pop("longitude", None)

    return doc


def generer_actions(offres: list, nom_index: str = NOM_INDEX):
    """
    Générateur d'actions pour l'insertion en bulk.

    bulk() d'Elasticsearch attend une liste d'actions formatées.
    On utilise l'id métier (ft_001, wttj_abc) comme _id Elasticsearch
    pour garantir l'unicité et permettre les mises à jour.
    """
    for offre in offres:
        doc = preparer_document(offre)
        yield {
            "_index": nom_index,
            "_id":    offre.get("id"),  # id métier = id Elasticsearch
            "_source": doc,
        }


# ─────────────────────────────────────────────────────────────
# INDEXATION EN BATCH
# ─────────────────────────────────────────────────────────────

def indexer_offres(
    offres:     list,
    client:     Elasticsearch,
    nom_index:  str = NOM_INDEX,
    chunk_size: int = 500,
) -> dict:
    """
    Indexe une liste d'offres dans Elasticsearch via l'API bulk.

    L'API bulk envoie plusieurs documents en une seule requête HTTP
    — beaucoup plus performant qu'un insert par document.

    chunk_size : nombre de documents par requête bulk (500 par défaut).
    Retourne un rapport d'indexation.
    """
    rapport = {"indexes": 0, "erreurs": 0, "detail_erreurs": []}

    if not offres:
        logging.warning("Liste vide — rien à indexer.")
        return rapport

    try:
        succes, erreurs = bulk(
            client,
            generer_actions(offres, nom_index),
            chunk_size=chunk_size,
            raise_on_error=False,   # ne pas planter sur les erreurs partielles
            stats_only=False,
        )

        rapport["indexes"] = succes
        rapport["erreurs"] = len(erreurs)
        rapport["detail_erreurs"] = erreurs

        logging.info(f"{succes} documents indexés, {len(erreurs)} erreurs.")

    except BulkIndexError as e:
        rapport["erreurs"] = len(e.errors)
        rapport["detail_erreurs"] = e.errors
        logging.error(f"Erreur bulk : {len(e.errors)} documents non indexés.")

    except Exception as e:
        logging.error(f"Erreur inattendue : {e}")
        raise

    return rapport


# ─────────────────────────────────────────────────────────────
# PIPELINE COMPLET
# ─────────────────────────────────────────────────────────────

def pipeline_indexation_elasticsearch() -> dict:
    """
    Pipeline complet :
        1. Connexion à Elasticsearch
        2. Création de l'index si absent
        3. Chargement des fichiers normalisés
        4. Indexation source par source
    """
    # sources = [
    #     ("FranceTravail", RACINE / "data" / "processed" / "francetravail"),
    #     ("WTTJ",          RACINE / "data" / "processed" / "welcometothejungle"),
    # ]
    sources = [("Offres normalisées", RACINE / "data" / "processed" / "normalise"),]
    rapport_final = {}

    # Connexion
    client = connecter_elasticsearch()

    # Créer l'index si absent
    creer_index(client)

    for nom_source, dossier in sources:
        print(f"\n=== Indexation Elasticsearch — {nom_source} ===")

        try:
            fichiers = sorted(dossier.glob("*.json"))
            if not fichiers:
                print(f"Aucun fichier trouvé pour {nom_source}")
                continue

            with open(fichiers[-1], "r", encoding="utf-8") as f:
                offres = json.load(f)

            print(f"{len(offres)} offres chargées")

            rapport = indexer_offres(offres, client)
            rapport_final[nom_source] = rapport

            print(f"Indexés  : {rapport['indexes']}")
            print(f"Erreurs  : {rapport['erreurs']}")

        except Exception as e:
            print(f"Erreur sur {nom_source} : {e}")
            rapport_final[nom_source] = {"erreur": str(e)}
            continue

    # # Vérification finale
    # total = client.count(index=NOM_INDEX)["count"]
    # print(f"\nTotal documents dans l'index '{NOM_INDEX}' : {total}")

    return rapport_final


# ─────────────────────────────────────────────────────────────
# POINT D'ENTRÉE
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    rapport = pipeline_indexation_elasticsearch()
    print("\n=== Rapport final ===")
    print(rapport)

2026-04-16 17:24:55,495 — INFO — HEAD http://localhost:9200/ [status:200 duration:0.004s]
2026-04-16 17:24:55,507 — INFO — GET http://localhost:9200/ [status:200 duration:0.011s]
2026-04-16 17:24:55,509 — INFO — Connecté à Elasticsearch v8.17.0 sur localhost:9200
2026-04-16 17:24:55,513 — INFO — HEAD http://localhost:9200/offres [status:404 duration:0.002s]
2026-04-16 17:24:55,790 — INFO — PUT http://localhost:9200/offres [status:200 duration:0.275s]
2026-04-16 17:24:55,791 — INFO — Index 'offres' créé avec succès.



=== Indexation Elasticsearch — Offres normalisées ===
120 offres chargées


2026-04-16 17:24:55,998 — INFO — PUT http://localhost:9200/_bulk [status:200 duration:0.193s]
2026-04-16 17:24:56,000 — INFO — 120 documents indexés, 0 erreurs.


Indexés  : 120
Erreurs  : 0

=== Rapport final ===
{'Offres normalisées': {'indexes': 120, 'erreurs': 0, 'detail_erreurs': []}}
